# Tetris in Jupyter Notebook

A complete playable game, generated with AI assistance, with the seven classic
pieces and an additional **white five-block arch**.

## Run the game

1. Open this file in Jupyter Notebook or JupyterLab with a **Python 3** kernel.
2. Choose **Run → Run All Cells** (called **Cell → Run All** in some versions).
3. Scroll to the game output, click **Start game**, and use the controls below.
   Click inside the game again if the notebook takes keyboard focus.

**No additional game packages or internet connection are needed.** The notebook
uses Python's standard library and IPython, which is included with a working
Python Jupyter kernel. The game renders directly in the browser.

If Jupyter is already installed, launch it from your notebook's folder with:

```bash
jupyter notebook
```

If that command is not recognized, try `python -m notebook` (or
`py -m notebook` on Windows). If the output is blank, run the last code cell
again; executing it locally makes the output trusted.


## Required additional piece

The white piece is a **five-block arch opening downward**: a row of three blocks
with one block immediately underneath each end. It uses the same movement,
rotation, collision, locking, line-clearing, hold, and preview rules as the
other pieces. Each shuffled eight-piece bag contains one of every piece,
including the white arch.

The next cell sets its shape. A `1` is a block; a `0` is an empty position.
The empty last row makes the matrix square so it can rotate.


In [1]:
# Additional white piece: three blocks on top, two underneath at the ends.
WHITE_SHAPE = [
    [1, 1, 1],
    [1, 0, 1],
    [0, 0, 0],
]
START_LEVEL = 1
SHOW_GHOST = True

assert len(WHITE_SHAPE) == 3 and all(len(row) == 3 for row in WHITE_SHAPE)
assert all(cell in (0, 1) for row in WHITE_SHAPE for cell in row)
assert sum(sum(row) for row in WHITE_SHAPE) == 5, "The white piece needs five blocks."
assert isinstance(START_LEVEL, int) and 1 <= START_LEVEL <= 20


In [2]:
# The entire game is embedded here: no downloads, servers, or external assets.
# Python displays the game; JavaScript handles real-time input in the browser.
GAME_DOCUMENT = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Tetris + one</title>
<style>
  :root { color-scheme: dark; --ink:#f1f4f6; --muted:#a6b3c0; --mint:#93e3c2;
    --surface:#1b2634; --line:#334253; }
  * { box-sizing:border-box; }
  body { margin:0; background:#101923; color:var(--ink);
    font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; }
  button { font:inherit; cursor:pointer; touch-action:manipulation; }
  button:focus-visible, #arcade:focus-visible { outline:2px solid var(--mint); outline-offset:3px; }
  #arcade { max-width:600px; margin:auto; padding:22px 24px 20px; outline:none; }
  header { display:flex; justify-content:space-between; align-items:center; gap:12px; margin-bottom:18px; }
  .eyebrow { margin:0 0 3px; color:var(--mint); font-size:10px; letter-spacing:2px; font-weight:750; }
  h1 { font-size:27px; letter-spacing:-1px; margin:0; font-weight:750; }
  .quiet-button { border:1px solid var(--line); color:var(--ink); background:#1b2634;
    padding:9px 12px; border-radius:8px; font-size:12px; }
  .quiet-button:hover { background:#293847; }
  .header-actions { display:flex; gap:7px; }
  .layout { display:grid; grid-template-columns:minmax(0,300px) minmax(0,1fr); gap:18px; align-items:start; }
  .well { position:relative; border:1px solid #354555; border-radius:10px; overflow:hidden;
    background:#111b26; box-shadow:0 14px 40px #0003; }
  #board { width:100%; height:auto; aspect-ratio:1 / 2; display:block; }
  .overlay { position:absolute; inset:0; z-index:2; background:#0d1625e8; display:flex;
    flex-direction:column; align-items:center; justify-content:center; text-align:center; padding:20px; }
  .overlay[hidden] { display:none; }
  .overlay .symbol { width:52px; height:52px; display:grid; place-items:center; border:1px solid #65847a;
    border-radius:16px; color:var(--mint); font-size:24px; margin-bottom:18px; background:#1e3432; }
  .overlay h2 { font-size:23px; letter-spacing:-.7px; margin:0 0 10px; }
  .overlay p { color:#b1bfce; font-size:13px; line-height:1.7; margin:0 0 22px; max-width:230px; }
  .primary { border:0; padding:12px 21px; border-radius:9px; color:#102b24; background:var(--mint);
    font-weight:750; font-size:13px; }
  .primary:hover { background:#b2f0d7; }
  .board-caption { display:flex; justify-content:space-between; align-items:center; margin:10px 1px 0;
    color:var(--muted); font-size:10px; letter-spacing:.6px; }
  .live-dot { display:inline-block; width:6px; height:6px; border-radius:50%; background:var(--mint); margin-right:6px; }
  #state-label { font-weight:650; color:#c9d9d6; }
  aside { display:flex; flex-direction:column; gap:12px; }
  .card { background:var(--surface); border:1px solid var(--line); border-radius:10px; padding:14px; }
  .label { text-transform:uppercase; letter-spacing:1.5px; font-size:10px; color:var(--muted); margin:0 0 7px; font-weight:700; }
  #score { font-size:33px; letter-spacing:-1px; line-height:1.2; font-weight:750; font-variant-numeric:tabular-nums; }
  .stats-row { display:grid; grid-template-columns:1fr 1fr; gap:10px; margin-top:16px; padding-top:12px; border-top:1px solid var(--line); }
  .stat-value { font-weight:700; font-size:18px; font-variant-numeric:tabular-nums; }
  .small-label { font-size:10px; color:var(--muted); display:block; margin-bottom:4px; }
  .preview { width:100%; display:block; }
  .hold-note { margin:5px 0 0; color:var(--muted); font-size:10px; }
  .white-card { background:#23332f; border-color:#4b645b; }
  .white-card .label { color:#acdac6; }
  .white-card p { font-size:11px; line-height:1.6; color:#c1d2cc; margin:4px 0 0; }
  .white-card strong { color:#f5f5f0; font-weight:650; }
  .touch-controls { display:grid; grid-template-columns:repeat(4,1fr) 1.15fr 1.15fr; gap:7px; margin-top:19px; }
  .control { background:#202e3c; border:1px solid #3a4a5a; color:var(--ink); border-radius:8px;
    min-height:40px; font-size:16px; padding:7px 4px; touch-action:none; user-select:none; }
  .control:hover { background:#2e4250; }
  .control:active { background:#40584e; border-color:var(--mint); }
  .control.word { font-size:12px; font-weight:650; }
  .control.drop { color:#b4efd7; border-color:#577665; }
  .keyboard-help { margin:13px 0 0; font-size:10px; line-height:1.8; text-align:center; color:var(--muted); }
  kbd { font-family:inherit; color:#d3dce5; font-size:10px; }
  .sr-only { position:absolute; width:1px; height:1px; padding:0; margin:-1px; overflow:hidden;
    clip:rect(0,0,0,0); white-space:nowrap; border:0; }
  @media (max-width:480px) {
    #arcade { padding:16px 12px; } .layout { gap:10px; grid-template-columns:minmax(0,1fr) 126px; }
    h1 { font-size:23px; } .card { padding:10px; } aside { gap:8px; } #score { font-size:26px; }
    .header-actions { gap:5px; } .quiet-button { padding:8px; font-size:10px; }
    .overlay h2 { font-size:18px; } .overlay p { font-size:11px; }
    .overlay { padding:12px; } .overlay .symbol { margin-bottom:12px; }
    .white-card p { font-size:10px; } .touch-controls { gap:5px; }
  }
</style>
</head>
<body>
<main id="arcade" tabindex="0" aria-label="Tetris game. Click here, then use the keyboard controls.">
  <header>
    <div><p class="eyebrow">JUPYTER ARCADE</p><h1>Tetris + one<span style="color:var(--mint)">.</span></h1></div>
    <div class="header-actions">
      <button type="button" class="quiet-button" data-action="pause" id="pause-button">Pause</button>
      <button type="button" class="quiet-button" data-action="restart">New game</button>
    </div>
  </header>
  <div class="layout">
    <section aria-label="Playfield">
      <div class="well">
        <canvas id="board" width="600" height="1200" aria-label="10 column, 20 row Tetris board"></canvas>
        <div class="overlay" id="overlay">
          <div class="symbol" id="overlay-symbol" aria-hidden="true">▥</div>
          <h2 id="overlay-title">Ready to stack?</h2>
          <p id="overlay-copy">Clear full rows and keep the board open. A new white arch joins the seven classic pieces.</p>
          <button type="button" id="overlay-button" class="primary" data-action="primary">Start game</button>
        </div>
      </div>
      <div class="board-caption"><span><span class="live-dot"></span><span id="state-label">READY</span></span><span>10 × 20</span></div>
    </section>
    <aside aria-label="Game information">
      <div class="card"><p class="label">Score</p><div id="score">0</div>
        <div class="stats-row"><div><span class="small-label">LEVEL</span><span class="stat-value" id="level">1</span></div>
          <div><span class="small-label">LINES</span><span class="stat-value" id="lines">0</span></div></div>
      </div>
      <div class="card"><p class="label">Up next</p><canvas class="preview" id="next" width="280" height="340" aria-label="Next three pieces"></canvas></div>
      <div class="card"><p class="label">Hold · C / Shift</p><canvas class="preview" id="held" width="280" height="130" aria-label="Held piece"></canvas>
        <p class="hold-note" id="hold-note">Once per falling piece</p></div>
      <div class="card white-card"><p class="label">The extra piece</p><canvas class="preview" id="white" width="280" height="120" aria-label="White five-block arch"></canvas>
        <p><strong>White arch · 5 blocks</strong><br>Part of every eight-piece bag.</p></div>
    </aside>
  </div>
  <div class="touch-controls" aria-label="On-screen controls">
    <button type="button" class="control" data-action="left" aria-label="Move left" title="Move left (Left arrow / A)">←</button>
    <button type="button" class="control" data-action="rotate" aria-label="Rotate clockwise" title="Rotate (Up arrow / W / X)">↻</button>
    <button type="button" class="control" data-action="right" aria-label="Move right" title="Move right (Right arrow / D)">→</button>
    <button type="button" class="control" data-action="down" aria-label="Soft drop" title="Soft drop (Down arrow / S)">↓</button>
    <button type="button" class="control word" data-action="hold" title="Hold (C / Shift)">Hold</button>
    <button type="button" class="control word drop" data-action="drop" title="Hard drop (Space)">Drop</button>
  </div>
  <p class="keyboard-help"><kbd>← →</kbd> move · <kbd>↑</kbd> rotate · <kbd>↓</kbd> soft drop · <kbd>Space</kbd> drop<br>
    <kbd>Z</kbd> rotate left · <kbd>C / Shift</kbd> hold · <kbd>P / Esc</kbd> pause · <kbd>R</kbd> new game</p>
  <div class="sr-only" id="announcement" aria-live="polite"></div>
</main>
<script>window.TETRIS_CONFIG = __TETRIS_CONFIGURATION__;</script>
<script>
/* Pure game rules. The renderer below uses the same engine as the checks. */
'use strict';

const COLS = 10;
const ROWS = 20;
const COLORS = {
  I: '#65daf0', J: '#6c91ef', L: '#f5af68', O: '#f4d36a',
  S: '#6dd5a0', T: '#bb95ee', Z: '#ef8095', W: '#f5f5f0'
};
const BASE_SHAPES = {
  I: [[0,0,0,0], [1,1,1,1], [0,0,0,0], [0,0,0,0]],
  J: [[1,0,0], [1,1,1], [0,0,0]],
  L: [[0,0,1], [1,1,1], [0,0,0]],
  O: [[1,1], [1,1]],
  S: [[0,1,1], [1,1,0], [0,0,0]],
  T: [[0,1,0], [1,1,1], [0,0,0]],
  Z: [[1,1,0], [0,1,1], [0,0,0]],
  // Required additional piece: a white, five-block arch, opening down.
  W: [[1,1,1], [1,0,1], [0,0,0]]
};
const cloneMatrix = matrix => matrix.map(row => row.slice());
function rotateMatrix(matrix, direction = 1) {
  const n = matrix.length;
  return Array.from({length: n}, (_, y) =>
    Array.from({length: n}, (_, x) =>
      direction === 1 ? matrix[n - 1 - x][y] : matrix[x][n - 1 - y]));
}

class TetrisGame {
  constructor(config = {}, rng = Math.random) {
    this.rng = rng;
    this.startLevel = config.startLevel || 1;
    this.shapes = Object.fromEntries(Object.entries(BASE_SHAPES)
      .map(([id, shape]) => [id, cloneMatrix(shape)]));
    if (config.whiteShape) this.shapes.W = cloneMatrix(config.whiteShape);
    this.reset();
  }

  reset() {
    this.board = Array.from({length: ROWS}, () => Array(COLS).fill(null));
    this.bag = [];
    this.queue = [];
    this.held = null;
    this.canHold = true;
    this.score = 0;
    this.lines = 0;
    this.level = this.startLevel;
    this.status = 'ready';
    this.clearFlash = 0;
    this.lastClear = 0;
    this.fillQueue();
    this.spawn();
  }

  start() { if (this.status === 'ready') this.status = 'playing'; }
  togglePause() {
    if (this.status === 'playing') this.status = 'paused';
    else if (this.status === 'paused') this.status = 'playing';
  }

  nextFromBag() {
    if (!this.bag.length) {
      this.bag = Object.keys(this.shapes);
      for (let i = this.bag.length - 1; i > 0; --i) {
        const j = Math.floor(this.rng() * (i + 1));
        [this.bag[i], this.bag[j]] = [this.bag[j], this.bag[i]];
      }
    }
    return this.bag.pop();
  }

  fillQueue() {
    while (this.queue.length < 5) this.queue.push(this.nextFromBag());
  }

  spawn(id = null) {
    if (!id) { id = this.queue.shift(); this.fillQueue(); }
    const matrix = cloneMatrix(this.shapes[id]);
    this.active = {id, matrix, x: Math.floor((COLS - matrix[0].length) / 2), y: 0};
    this.dropTime = 0;
    this.lockTime = 0;
    this.lockResets = 0;
    if (this.collides(matrix, this.active.x, this.active.y)) this.status = 'over';
  }

  collides(matrix, px, py) {
    for (let y = 0; y < matrix.length; ++y) {
      for (let x = 0; x < matrix[y].length; ++x) {
        if (!matrix[y][x]) continue;
        const bx = px + x, by = py + y;
        if (bx < 0 || bx >= COLS || by >= ROWS) return true;
        if (by >= 0 && this.board[by][bx] !== null) return true;
      }
    }
    return false;
  }

  grounded() {
    const a = this.active;
    return this.collides(a.matrix, a.x, a.y + 1);
  }

  afterAdjustment(wasGrounded) {
    if (wasGrounded && this.lockResets < 15) {
      this.lockTime = 0;
      ++this.lockResets;
    } else if (!this.grounded()) this.lockTime = 0;
  }

  move(dx, dy = 0) {
    if (this.status !== 'playing') return false;
    const a = this.active;
    if (this.collides(a.matrix, a.x + dx, a.y + dy)) return false;
    const wasGrounded = this.grounded();
    a.x += dx;
    a.y += dy;
    this.afterAdjustment(wasGrounded);
    return true;
  }

  rotate(direction = 1) {
    if (this.status !== 'playing' || this.active.id === 'O') return false;
    const a = this.active, turned = rotateMatrix(a.matrix, direction);
    const wasGrounded = this.grounded();
    // Small horizontal/upward wall kicks also support the extra white piece.
    const kicks = [[0,0], [-1,0], [1,0], [-2,0], [2,0],
                   [0,-1], [-1,-1], [1,-1], [0,-2]];
    for (const [dx, dy] of kicks) {
      if (!this.collides(turned, a.x + dx, a.y + dy)) {
        a.matrix = turned;
        a.x += dx;
        a.y += dy;
        this.afterAdjustment(wasGrounded);
        return true;
      }
    }
    return false;
  }

  ghostY() {
    const a = this.active;
    let y = a.y;
    while (!this.collides(a.matrix, a.x, y + 1)) ++y;
    return y;
  }

  softDrop() {
    if (this.move(0, 1)) { ++this.score; return true; }
    return false;
  }

  hardDrop() {
    if (this.status !== 'playing') return 0;
    const distance = this.ghostY() - this.active.y;
    this.active.y += distance;
    this.score += 2 * distance;
    this.lock();
    return distance;
  }

  hold() {
    if (this.status !== 'playing' || !this.canHold) return false;
    const previous = this.held;
    this.held = this.active.id;
    this.spawn(previous);
    this.canHold = false;
    return true;
  }

  clearLines() {
    const remaining = this.board.filter(row => row.some(cell => cell === null));
    const removed = ROWS - remaining.length;
    this.board = [
      ...Array.from({length: removed}, () => Array(COLS).fill(null)),
      ...remaining
    ];
    if (removed) {
      const linePoints = [0,100,300,500,800];
      this.score += (linePoints[removed] || (800 + (removed - 4) * 300)) * this.level;
      this.lines += removed;
      this.level = this.startLevel + Math.floor(this.lines / 10);
      this.clearFlash = 180;
    }
    this.lastClear = removed;
    return removed;
  }

  lock() {
    const a = this.active;
    const cells = [];
    for (let y = 0; y < a.matrix.length; ++y) {
      for (let x = 0; x < a.matrix[y].length; ++x) {
        if (!a.matrix[y][x]) continue;
        if (a.y + y < 0) { this.status = 'over'; return; }
        cells.push([a.x + x, a.y + y]);
      }
    }
    for (const [x, y] of cells) this.board[y][x] = a.id;
    this.clearLines();
    this.canHold = true;
    this.spawn();
  }

  dropInterval() { return Math.max(70, 900 * Math.pow(0.82, this.level - 1)); }

  update(milliseconds, softDropping = false) {
    if (this.status !== 'playing') return;
    this.clearFlash = Math.max(0, this.clearFlash - milliseconds);
    this.dropTime += milliseconds;
    const interval = softDropping ? 35 : this.dropInterval();
    while (this.dropTime >= interval) {
      this.dropTime -= interval;
      const moved = this.move(0, 1);
      if (softDropping && moved) ++this.score;
      if (!moved) { this.dropTime = 0; break; }
    }
    if (this.grounded()) {
      this.lockTime += milliseconds;
      if (this.lockTime >= 500) this.lock();
    } else this.lockTime = 0;
  }
}

if (typeof module !== 'undefined' && module.exports) {
  module.exports = {TetrisGame, rotateMatrix, BASE_SHAPES, COLORS, COLS, ROWS};
}

</script>
<script>
'use strict';
const config = window.TETRIS_CONFIG;
const game = new TetrisGame(config);
window.TetrisGame = TetrisGame;
window.tetris = game;
const root = document.getElementById('arcade');
const boardCanvas = document.getElementById('board');
const boardContext = boardCanvas.getContext('2d');
const overlay = document.getElementById('overlay');
const keyHeld = new Set();
const pointerHeld = new Set();
const cellSize = 60;
let lastTime = 0, horizontalAction = null, horizontalTime = 0, horizontalRepeating = false;
let previousStatus = '', previousLines = -1;

function block(ctx, px, py, size, color, ghost = false) {
  const inset = size * 0.075;
  if (ghost) {
    ctx.strokeStyle = color;
    ctx.globalAlpha = .5;
    ctx.lineWidth = Math.max(1, size * .045);
    ctx.strokeRect(px + inset, py + inset, size - 2 * inset, size - 2 * inset);
    ctx.globalAlpha = 1;
    return;
  }
  ctx.fillStyle = color;
  ctx.fillRect(px + inset, py + inset, size - 2 * inset, size - 2 * inset);
  ctx.fillStyle = '#ffffff38';
  ctx.fillRect(px + inset, py + inset, size - 2 * inset, size * .07);
  ctx.fillStyle = '#00000020';
  ctx.fillRect(px + inset, py + size * .84, size - 2 * inset, size * .085);
}

function drawActive(atY, ghost = false) {
  const a = game.active;
  for (let y = 0; y < a.matrix.length; ++y)
    for (let x = 0; x < a.matrix[y].length; ++x)
      if (a.matrix[y][x] && atY + y >= 0)
        block(boardContext, (a.x + x) * cellSize, (atY + y) * cellSize, cellSize, COLORS[a.id], ghost);
}

function occupiedBounds(matrix) {
  const cells = [];
  matrix.forEach((row, y) => row.forEach((value, x) => { if (value) cells.push([x,y]); }));
  return {minX:Math.min(...cells.map(c=>c[0])), maxX:Math.max(...cells.map(c=>c[0])),
          minY:Math.min(...cells.map(c=>c[1])), maxY:Math.max(...cells.map(c=>c[1]))};
}

function previewPiece(ctx, id, top, height, size = 42, dim = false) {
  if (!id) return;
  const matrix = game.shapes[id], bounds = occupiedBounds(matrix);
  const width = (bounds.maxX - bounds.minX + 1) * size;
  const pieceHeight = (bounds.maxY - bounds.minY + 1) * size;
  const startX = (ctx.canvas.width - width) / 2;
  const startY = top + (height - pieceHeight) / 2;
  if (dim) ctx.globalAlpha = .4;
  matrix.forEach((row,y) => row.forEach((value,x) => {
    if (value) block(ctx, startX + (x-bounds.minX)*size, startY + (y-bounds.minY)*size, size, COLORS[id]);
  }));
  ctx.globalAlpha = 1;
}

function drawPreviews() {
  const next = document.getElementById('next').getContext('2d');
  next.clearRect(0,0,next.canvas.width,next.canvas.height);
  game.queue.slice(0,3).forEach((id,i) => previewPiece(next,id,i*110,110,38));
  const held = document.getElementById('held').getContext('2d');
  held.clearRect(0,0,held.canvas.width,held.canvas.height);
  if (game.held) previewPiece(held,game.held,0,130,42,!game.canHold);
  else {
    held.fillStyle = '#718397'; held.font = '24px system-ui'; held.textAlign = 'center';
    held.fillText('empty',140,76);
  }
  const white = document.getElementById('white').getContext('2d');
  white.clearRect(0,0,white.canvas.width,white.canvas.height);
  previewPiece(white,'W',0,120,43);
}

function setText(id, text) { document.getElementById(id).textContent = text; }
function releaseInputs() {
  keyHeld.clear(); pointerHeld.clear(); horizontalAction = null;
  horizontalTime = 0; horizontalRepeating = false;
}

function syncStatus() {
  setText('score',game.score.toLocaleString());
  setText('level',game.level); setText('lines',game.lines);
  setText('hold-note',game.canHold ? 'Once per falling piece' : 'Available after this piece locks');
  setText('pause-button',game.status === 'paused' ? 'Resume' : 'Pause');
  setText('state-label',({ready:'READY',playing:'PLAYING',paused:'PAUSED',over:'GAME OVER'})[game.status]);
  if (game.lines !== previousLines) {
    if (previousLines >= 0 && game.lines > previousLines)
      setText('announcement',`${game.lines-previousLines} lines cleared. Score ${game.score}.`);
    previousLines = game.lines;
  }
  if (game.status === previousStatus) return;
  previousStatus = game.status;
  overlay.hidden = game.status === 'playing';
  if (game.status === 'playing') return;
  releaseInputs();
  const content = {
    ready:['▥','Ready to stack?','Clear full rows and keep the board open. A new white arch joins the seven classic pieces.','Start game'],
    paused:['Ⅱ','Paused','Take your time. Your board will be right here.','Resume game'],
    over:['↺','Game over',`You cleared ${game.lines} lines and scored ${game.score.toLocaleString()} points. Give it another go.`,'Play again']
  }[game.status];
  ['overlay-symbol','overlay-title','overlay-copy','overlay-button'].forEach((id,i)=>setText(id,content[i]));
  setText('announcement',content[1]);
}

function render() {
  boardContext.fillStyle = '#111b26';
  boardContext.fillRect(0,0,boardCanvas.width,boardCanvas.height);
  boardContext.strokeStyle = '#243240'; boardContext.lineWidth = 1;
  for (let x = 0; x <= COLS; ++x) {
    boardContext.beginPath(); boardContext.moveTo(x*cellSize,0);
    boardContext.lineTo(x*cellSize,ROWS*cellSize); boardContext.stroke();
  }
  for (let y = 0; y <= ROWS; ++y) {
    boardContext.beginPath(); boardContext.moveTo(0,y*cellSize);
    boardContext.lineTo(COLS*cellSize,y*cellSize); boardContext.stroke();
  }
  game.board.forEach((row,y)=>row.forEach((id,x)=>{
    if (id) block(boardContext,x*cellSize,y*cellSize,cellSize,COLORS[id]);
  }));
  if (game.status !== 'over') {
    if (config.ghostEnabled && game.ghostY() > game.active.y) drawActive(game.ghostY(),true);
    drawActive(game.active.y);
  }
  if (game.clearFlash) {
    boardContext.fillStyle = `rgba(147,227,194,${game.clearFlash/180*.12})`;
    boardContext.fillRect(0,0,boardCanvas.width,boardCanvas.height);
  }
  drawPreviews(); syncStatus();
}

function takeAction(action) {
  if (action === 'primary') {
    if (game.status === 'ready') game.start();
    else if (game.status === 'paused') game.togglePause();
    else if (game.status === 'over') { game.reset(); game.start(); }
  } else if (action === 'restart') { releaseInputs(); game.reset(); game.start(); }
  else if (action === 'pause') { game.togglePause(); releaseInputs(); }
  else if (game.status === 'playing') {
    if (action === 'left') game.move(-1);
    else if (action === 'right') game.move(1);
    else if (action === 'rotate') game.rotate(1);
    else if (action === 'counter') game.rotate(-1);
    else if (action === 'down') game.softDrop();
    else if (action === 'drop') game.hardDrop();
    else if (action === 'hold') game.hold();
  }
  render();
}

const keyMap = {
  ArrowLeft:'left', KeyA:'left', ArrowRight:'right', KeyD:'right', ArrowDown:'down', KeyS:'down',
  ArrowUp:'rotate', KeyW:'rotate', KeyX:'rotate', KeyZ:'counter', Space:'drop',
  KeyC:'hold', ShiftLeft:'hold', ShiftRight:'hold', KeyP:'pause', Escape:'pause', KeyR:'restart'
};
root.addEventListener('keydown',event=>{
  if (event.ctrlKey || event.metaKey || event.altKey) return;
  if (event.target instanceof HTMLButtonElement && (event.code === 'Space' || event.code === 'Enter')) return;
  const action = keyMap[event.code];
  if (!action) return;
  event.preventDefault();
  if (event.repeat || keyHeld.has(event.code)) return;
  keyHeld.add(event.code);
  if (action === 'left' || action === 'right') {
    horizontalAction = action; horizontalTime = 0; horizontalRepeating = false;
  }
  takeAction(action);
});
root.addEventListener('keyup',event=>{
  keyHeld.delete(event.code);
  if (keyMap[event.code] === horizontalAction) {
    horizontalAction = null; horizontalTime = 0; horizontalRepeating = false;
  }
});

root.querySelectorAll('button[data-action]').forEach(button=>{
  const action = button.dataset.action;
  const repeats = ['left','right','down'].includes(action);
  if (repeats) {
    button.addEventListener('pointerdown',event=>{
      event.preventDefault(); root.focus({preventScroll:true});
      button.setPointerCapture(event.pointerId); pointerHeld.add(action);
      if (action !== 'down') { horizontalAction = action; horizontalTime = 0; horizontalRepeating = false; }
      takeAction(action);
    });
    const finish=()=>{
      pointerHeld.delete(action);
      if (horizontalAction === action) { horizontalAction=null; horizontalTime=0; horizontalRepeating=false; }
    };
    button.addEventListener('pointerup',finish);
    button.addEventListener('pointercancel',finish);
    button.addEventListener('lostpointercapture',finish);
    button.addEventListener('click',event=>{
      // Keyboard activation has detail 0; pointer activation was handled above.
      if (event.detail === 0) { takeAction(action); root.focus({preventScroll:true}); }
    });
  } else button.addEventListener('click',()=>{ takeAction(action); root.focus({preventScroll:true}); });
});
boardCanvas.addEventListener('pointerdown',()=>root.focus({preventScroll:true}));
root.addEventListener('focusout',event=>{
  if (!root.contains(event.relatedTarget)) releaseInputs();
});
function autoPause() {
  releaseInputs();
  if (game.status === 'playing') { game.togglePause(); render(); }
}
window.addEventListener('blur',autoPause);
document.addEventListener('visibilitychange',()=>{ if (document.hidden) autoPause(); });

function animate(timestamp) {
  const elapsed = lastTime ? Math.min(timestamp-lastTime,80) : 0;
  lastTime = timestamp;
  if (game.status === 'playing') {
    if (horizontalAction) {
      horizontalTime += elapsed;
      const delay = horizontalRepeating ? 55 : 165;
      if (horizontalTime >= delay) {
        horizontalTime -= delay; horizontalRepeating = true;
        takeAction(horizontalAction);
      }
    }
    const soft = keyHeld.has('ArrowDown') || keyHeld.has('KeyS') || pointerHeld.has('down');
    game.update(elapsed,soft);
  }
  render(); requestAnimationFrame(animate);
}
render(); requestAnimationFrame(animate);
</script>
</body>
</html>
'''


In [3]:
import html
import json
from IPython.display import HTML, display

game_config = {
    "whiteShape": WHITE_SHAPE,
    "startLevel": START_LEVEL,
    "ghostEnabled": SHOW_GHOST,
}
game_html = GAME_DOCUMENT.replace(
    "__TETRIS_CONFIGURATION__", json.dumps(game_config)
)

# An isolated frame keeps game keys and styles separate from the notebook.
frame = (
    '<iframe title="Tetris with the extra white arch piece" '
    'sandbox="allow-scripts" '
    'style="width:100%;height:910px;border:0;border-radius:14px;display:block;" '
    f'srcdoc="{html.escape(game_html, quote=True)}"></iframe>'
)
display(HTML(frame))


/usr/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Controls and scoring

| Action | Keyboard |
|---|---|
| Move left / right | Left / Right arrow, or A / D |
| Rotate clockwise | Up arrow, W, or X |
| Rotate counterclockwise | Z |
| Soft drop | Hold Down arrow or S |
| Hard drop and lock | Space |
| Hold / swap a piece | C or Shift; once per falling piece |
| Pause / resume | P or Escape |
| Restart | R, or the New game button |

The buttons below the board also work with a mouse or touch. Holding a movement
button repeats its action. The outline shows where the active piece will land.
The game pauses automatically when its browser frame loses focus.

Clearing 1, 2, 3, or 4 rows gives **100, 300, 500, or 800 points × the current
level**, respectively. Soft drop adds 1 point per row moved; hard drop adds 2.
Every 10 cleared rows increases the level and falling speed. A piece locks
after resting for about half a second, or immediately on a hard drop.
The game ends when a new piece cannot enter the playfield or a piece locks
above it.

To change the starting level or turn off the landing outline, edit the short
configuration cell and run all cells again. Running the display cell again
starts a fresh game.
